# Análisis exploratorio de series — productos

Notebook de exploración, **independiente del pipe**. Lee los crudos del bucket
(`sell-in.txt.gz` + `tb_productos.txt`) y arma su propio panel producto-mes, así que
no depende de qué combinación de palancas hayas corrido en `01_Preprocesamiento`.

Qué responde:

1. **Panorama** — cuántos productos, cuánto concentra el top, estacionalidad del mercado.
2. **Ciclo de vida** — lanzamiento → crecimiento → pico → caída → estabilización → declive,
   clasificado automáticamente mes a mes.
3. **Productos similares** — qué series se parecen y si eso coincide con la jerarquía
   de categorías.
4. **Canibalización** — cuando entra un producto nuevo, ¿le come ventas a los de su
   misma categoría? Se mide con un *event study* alineado al mes de lanzamiento.
5. **Complementarios** — productos que los mismos clientes compran juntos más de lo
   que se esperaría por azar.
6. **Features exportables** — todo lo anterior termina en un parquet que se puede
   pegar al dataset de `02_FE`.

> Una advertencia que atraviesa todo el notebook: los productos que ya existían en
> **2017-01** están *censurados a izquierda* — no sabemos cuándo nacieron ni en qué
> fase están. Todo análisis de ciclo de vida los separa explícitamente.

## 0 — Ambiente, paleta y carga

In [ ]:
import os, json, itertools
from pathlib import Path

import numpy as np
import polars as pl
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter


def resolver_bucket() -> Path:
    env = os.environ.get("LABO3_BUCKET")
    if env:
        return Path(env).expanduser().resolve()
    # ~/buckets/b1 primero: es donde lo monta la instalacion de la catedra,
    # y sirve para cualquier usuario (ds, natalialabo3, el que sea).
    for cand in (Path.home() / "buckets" / "b1",
                 "/content/buckets/b1",
                 "/home/ds/buckets/b1"):
        if Path(cand).is_dir():
            return Path(cand)
    raise RuntimeError(
        "No encontre el bucket. Defini LABO3_BUCKET, ej. "
        "os.environ['LABO3_BUCKET'] = '/ruta/al/bucket'"
    )


BUCKET  = resolver_bucket()
DIR_RAW = BUCKET / "datasets"
DIR_OUT = BUCKET / "datasets_fe"
DIR_OUT.mkdir(parents=True, exist_ok=True)
print(f"BUCKET: {BUCKET}")

In [ ]:
# ── Paleta ────────────────────────────────────────────────────────────────
# Orden categorico FIJO (nunca se cicla): si hicieran falta mas de 8 series,
# se usan small multiples o se agrupa en "Otros", no se inventan colores.
SERIE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100",
         "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
SEQ   = ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"]
DIV_NEG, DIV_MID, DIV_POS = "#e34948", "#f0efec", "#2a78d6"   # divergente rojo<->azul

TINTA, TINTA2, MUDO = "#0b0b0b", "#52514e", "#898781"
GRILLA, EJE, FONDO  = "#e1e0d9", "#c3c2b7", "#fcfcfb"

plt.rcParams.update({
    "figure.facecolor": FONDO, "axes.facecolor": FONDO,
    "axes.edgecolor": EJE, "axes.labelcolor": TINTA2,
    "text.color": TINTA, "xtick.color": MUDO, "ytick.color": MUDO,
    "grid.color": GRILLA, "grid.linewidth": 0.8,
    "axes.grid": True, "axes.axisbelow": True,
    "axes.spines.top": False, "axes.spines.right": False,
    "font.size": 9, "axes.titlesize": 10, "figure.dpi": 110,
    "legend.frameon": False,
})


def limpiar(ax, titulo=None, y=None, x=None):
    """Cromo minimo y consistente para cada eje."""
    if titulo: ax.set_title(titulo, color=TINTA, loc="left", pad=10)
    if y: ax.set_ylabel(y)
    if x: ax.set_xlabel(x)
    ax.grid(axis="x", visible=False)
    return ax


def miles(ax):
    ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:,.0f}".replace(",", ".")))


print("paleta lista")

### Panel producto-mes

`periodo` se maneja como **índice de mes entero** (`m = año*12 + mes`) en vez de fecha:
los lags, las edades y los alineamientos son restas, y así un `shift` de k filas es
siempre k meses. Se guarda también `periodo` en AAAAMM para leerlo.

In [ ]:
sell = pl.read_csv(DIR_RAW / "sell-in.txt.gz", separator="\t")
prod = pl.read_csv(DIR_RAW / "tb_productos.txt", separator="\t").unique(subset=["product_id"])
print(f"sell-in : {sell.shape}")
print(f"productos: {prod.shape}   columnas: {prod.columns}")

def a_m(col):     # AAAAMM -> indice de mes absoluto
    return (pl.col(col) // 100) * 12 + (pl.col(col) % 100)

# Panel producto-mes: se colapsa la dimension cliente sumando tn.
panel = (
    sell.group_by(["product_id", "periodo"])
        .agg(pl.col("tn").sum().alias("tn"),
             pl.col("customer_id").n_unique().alias("n_clientes"))
        .with_columns(a_m("periodo").alias("m"))
        .sort(["product_id", "m"])
)

M_MIN, M_MAX = panel["m"].min(), panel["m"].max()
def m_a_periodo(m):  return ((m - 1) // 12) * 100 + ((m - 1) % 12) + 1

print(f"\npanel: {panel.height:,} filas · {panel['product_id'].n_unique()} productos")
print(f"rango: {m_a_periodo(M_MIN)} -> {m_a_periodo(M_MAX)}  ({M_MAX-M_MIN+1} meses)")

In [ ]:
# Densificacion: dentro de la vida de cada producto, un mes sin registro es una venta
# de 0 tn, no un dato faltante. Sin esto las caidas a cero desaparecen y el ciclo de
# vida se lee mal.
vida = panel.group_by("product_id").agg(
    pl.col("m").min().alias("m_nace"),
    pl.col("m").max().alias("m_muere"),
    pl.col("tn").sum().alias("tn_total"),
)

grilla = (
    vida.select("product_id", "m_nace", "m_muere")
        .with_columns(pl.int_ranges("m_nace", pl.col("m_muere") + 1).alias("m"))
        .explode("m")
        .drop("m_nace", "m_muere")
)

panel = (
    grilla.join(panel, on=["product_id", "m"], how="left")
          .with_columns(pl.col("tn").fill_null(0.0),
                        pl.col("n_clientes").fill_null(0))
          .with_columns(pl.col("m").map_elements(m_a_periodo, return_dtype=pl.Int64).alias("periodo"))
          .join(prod.select("product_id", "cat1", "cat2", "cat3", "brand"),
                on="product_id", how="left")
          .join(vida.select("product_id", "m_nace", "m_muere", "tn_total"),
                on="product_id", how="left")
          .with_columns((pl.col("m") - pl.col("m_nace")).alias("edad"),
                        (pl.col("m_muere") - pl.col("m_nace") + 1).alias("largo_vida"))
          .sort(["product_id", "m"])
)

# Censura a izquierda / derecha: no sabemos la edad real de los que ya existian al
# inicio de la ventana, ni el final de los que siguen vivos al cierre.
panel = panel.with_columns(
    (pl.col("m_nace") == M_MIN).alias("censura_izq"),
    (pl.col("m_muere") == M_MAX).alias("censura_der"),
)

n_izq = panel.filter(pl.col("censura_izq"))["product_id"].n_unique()
n_tot = panel["product_id"].n_unique()
print(f"panel densificado: {panel.height:,} filas")
print(f"productos censurados a izquierda (ya existian en {m_a_periodo(M_MIN)}): "
      f"{n_izq} de {n_tot} ({100*n_izq/n_tot:.0f}%)")
print(f"productos nacidos DENTRO de la ventana: {n_tot - n_izq}  <- los unicos con edad real")

## 1 — Panorama del mercado

In [ ]:
serie_total = panel.group_by("m").agg(pl.col("tn").sum().alias("tn")).sort("m")
x = serie_total["m"].to_numpy(); y = serie_total["tn"].to_numpy()

fig, ax = plt.subplots(figsize=(11, 3.4))
ax.plot(x, y, color=SERIE[0], linewidth=2)
ax.fill_between(x, 0, y, color=SERIE[0], alpha=.08)
ticks = [m for m in x if m_a_periodo(m) % 100 in (1, 7)]
ax.set_xticks(ticks); ax.set_xticklabels([str(m_a_periodo(m)) for m in ticks], rotation=45, ha="right")
ax.set_ylim(0, None); miles(ax)
limpiar(ax, "Toneladas totales del mercado por mes", y="tn")
# Etiqueta directa en vez de leyenda: una sola serie no necesita caja de leyenda.
ax.annotate(f"{y[-1]:,.0f}".replace(",", "."), (x[-1], y[-1]), xytext=(6, 0),
            textcoords="offset points", va="center", color=TINTA2, fontsize=9)
plt.tight_layout(); plt.show()

print(f"tn total del periodo : {y.sum():,.0f}".replace(",", "."))
print(f"promedio mensual     : {y.mean():,.0f}".replace(",", "."))
print(f"mes maximo / minimo  : {m_a_periodo(x[y.argmax()])} / {m_a_periodo(x[y.argmin()])}")

In [ ]:
# Concentracion: que porcion de los productos hace el grueso del volumen.
v = vida.sort("tn_total", descending=True)
acum = np.cumsum(v["tn_total"].to_numpy()); acum = 100 * acum / acum[-1]
pct_prod = 100 * np.arange(1, len(acum) + 1) / len(acum)
n80 = int(np.searchsorted(acum, 80) + 1)

fig, ax = plt.subplots(figsize=(6.5, 3.6))
ax.plot(pct_prod, acum, color=SERIE[0], linewidth=2)
ax.plot([0, 100], [0, 100], color=MUDO, linewidth=1, linestyle=(0, (4, 3)))
ax.axhline(80, color=EJE, linewidth=1)
ax.axvline(100*n80/len(acum), color=SERIE[1], linewidth=1.5)
ax.annotate(f"{n80} productos ({100*n80/len(acum):.0f}%)\\nhacen el 80% del volumen",
            (100*n80/len(acum), 80), xytext=(10, -34), textcoords="offset points",
            color=TINTA2, fontsize=9)
ax.set_xlim(0, 100); ax.set_ylim(0, 100)
limpiar(ax, "Concentración del volumen", y="% acumulado de tn", x="% de productos (ordenados por volumen)")
plt.tight_layout(); plt.show()

print(f"{len(acum)} productos · el top {n80} concentra el 80% de las toneladas")

In [ ]:
# Estacionalidad: indice mensual promedio (100 = mes tipico), quitando la tendencia
# con una media movil centrada de 12 meses.
s = serie_total.with_columns(
    pl.col("tn").rolling_mean(window_size=12, center=True, min_periods=12).alias("tendencia")
).drop_nulls()
s = s.with_columns((100 * pl.col("tn") / pl.col("tendencia")).alias("indice"),
                   ((pl.col("m") - 1) % 12 + 1).alias("mes_cal"))
est = s.group_by("mes_cal").agg(pl.col("indice").mean()).sort("mes_cal")

fig, ax = plt.subplots(figsize=(7, 3.2))
vals = est["indice"].to_numpy()
cols = [SERIE[0] if v >= 100 else SERIE[1] for v in vals]
ax.bar(est["mes_cal"], vals - 100, bottom=100, color=cols, width=.68)
ax.axhline(100, color=EJE, linewidth=1.2)
for mm, vv in zip(est["mes_cal"], vals):
    ax.annotate(f"{vv:.0f}", (mm, vv), xytext=(0, 3 if vv >= 100 else -12),
                textcoords="offset points", ha="center", color=TINTA2, fontsize=8)
ax.set_xticks(range(1, 13))
ax.set_xticklabels(["E","F","M","A","M","J","J","A","S","O","N","D"])
limpiar(ax, "Índice de estacionalidad (100 = mes típico)", y="índice")
plt.tight_layout(); plt.show()

## 2 — Ciclo de vida del producto

La idea: cada producto tiene una historia propia — arranca, crece, llega a un pico,
cae, y a veces se estabiliza o muere. Para compararlos entre sí hay que sacarles la
escala (un producto grande y uno chico pueden tener la misma **forma**) y alinearlos
por **edad** en vez de por fecha.

Dos normalizaciones distintas, para dos preguntas distintas:

- `ratio_vs_pico = tn / max(tn del producto)` → forma de la curva, en 0–1.
- `edad = meses desde el primer mes con venta` → eje temporal propio de cada producto.

**Sólo los productos nacidos dentro de la ventana tienen edad real.** Los censurados a
izquierda se excluyen de las curvas promedio, y se marcan cuando aparecen.

In [ ]:
# Suavizado de 3 meses para que el pico no lo defina un mes aislado.
panel = panel.with_columns(
    pl.col("tn").rolling_mean(3, min_periods=1).over("product_id").alias("tn_suave")
)

pico = (panel.sort(["product_id", "tn_suave"], descending=[False, True])
             .group_by("product_id").first()
             .select("product_id",
                     pl.col("m").alias("m_pico"),
                     pl.col("tn_suave").alias("tn_pico"),
                     pl.col("edad").alias("edad_pico")))

panel = (panel.join(pico, on="product_id", how="left")
              .with_columns(
                  (pl.col("m") - pl.col("m_pico")).alias("meses_desde_pico"),
                  pl.when(pl.col("tn_pico") > 0)
                    .then(pl.col("tn_suave") / pl.col("tn_pico"))
                    .otherwise(0.0).alias("ratio_vs_pico"),
              ))

# Pendiente relativa a 3 meses: cuanto cambio la serie suavizada respecto de su
# propio pico (asi es comparable entre productos de distinta escala).
panel = panel.with_columns(
    ((pl.col("tn_suave") - pl.col("tn_suave").shift(3).over("product_id"))
     / pl.when(pl.col("tn_pico") > 0).then(pl.col("tn_pico")).otherwise(1.0)
    ).alias("pend3")
)
print("pico y pendiente calculados")

In [ ]:
# ── Clasificacion de fase ────────────────────────────────────────────────
# Reglas en cascada (la primera que matchea gana). UMBRAL es el cambio relativo
# a 3 meses, medido en fracciones del pico del propio producto.
UMBRAL = 0.10

panel = panel.with_columns(
    pl.when(pl.col("edad") <= 2)
      .then(pl.lit("1_lanzamiento"))
    .when((pl.col("m") < pl.col("m_pico")) & (pl.col("pend3") > UMBRAL))
      .then(pl.lit("2_crecimiento"))
    .when(pl.col("meses_desde_pico").abs() <= 1)
      .then(pl.lit("3_pico"))
    .when((pl.col("m") > pl.col("m_pico")) & (pl.col("pend3") < -UMBRAL))
      .then(pl.lit("4_caida"))
    .when(pl.col("pend3").abs() <= UMBRAL)
      .then(pl.lit("5_estabilizacion"))
    .when(pl.col("pend3") < -UMBRAL)
      .then(pl.lit("6_declive"))
    .otherwise(pl.lit("2_crecimiento"))
    .alias("fase")
)

FASES = ["1_lanzamiento", "2_crecimiento", "3_pico", "4_caida", "5_estabilizacion", "6_declive"]
COLOR_FASE = dict(zip(FASES, SERIE[:6]))

rep = (panel.group_by("fase").agg(pl.len().alias("filas"), pl.col("tn").sum().alias("tn"))
            .sort("fase")
            .with_columns((100 * pl.col("filas") / panel.height).round(1).alias("%_filas"),
                          (100 * pl.col("tn") / panel["tn"].sum()).round(1).alias("%_tn")))
print(rep)

In [ ]:
# Composicion de fases mes a mes (area apilada). Se usan slots categoricos en orden
# fijo y se etiqueta cada banda: la identidad nunca queda solo en el color.
comp = (panel.group_by(["m", "fase"]).agg(pl.col("product_id").n_unique().alias("n"))
             .pivot(on="fase", index="m", values="n").fill_null(0).sort("m"))
xs = comp["m"].to_numpy()
capas = [comp[f].to_numpy() if f in comp.columns else np.zeros(len(xs)) for f in FASES]
tot = np.maximum(np.sum(capas, axis=0), 1)
capas_pct = [100 * c / tot for c in capas]

fig, ax = plt.subplots(figsize=(11, 3.8))
ax.stackplot(xs, *capas_pct, colors=[COLOR_FASE[f] for f in FASES],
             labels=[f[2:] for f in FASES], edgecolor=FONDO, linewidth=1.2)
ticks = [m for m in xs if m_a_periodo(m) % 100 in (1, 7)]
ax.set_xticks(ticks); ax.set_xticklabels([str(m_a_periodo(m)) for m in ticks], rotation=45, ha="right")
ax.set_ylim(0, 100); ax.set_xlim(xs.min(), xs.max())
limpiar(ax, "Composición de la cartera por fase del ciclo de vida", y="% de productos vivos")
ax.legend(loc="upper center", bbox_to_anchor=(.5, -.28), ncol=6, fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
# ── Curva de vida promedio ───────────────────────────────────────────────
# Solo productos NACIDOS dentro de la ventana (edad conocida) y con al menos 12
# meses de historia, para que la mediana no la domine el ruido de los efimeros.
cohorte = (panel.filter(~pl.col("censura_izq") & (pl.col("largo_vida") >= 12)))
n_coh = cohorte["product_id"].n_unique()

curva = (cohorte.group_by("edad")
                .agg(pl.col("ratio_vs_pico").median().alias("mediana"),
                     pl.col("ratio_vs_pico").quantile(.25).alias("q25"),
                     pl.col("ratio_vs_pico").quantile(.75).alias("q75"),
                     pl.col("product_id").n_unique().alias("n"))
                .filter(pl.col("n") >= 10).sort("edad"))

e = curva["edad"].to_numpy()
fig, ax = plt.subplots(figsize=(8, 3.8))
ax.fill_between(e, curva["q25"], curva["q75"], color=SERIE[0], alpha=.15)
ax.plot(e, curva["mediana"], color=SERIE[0], linewidth=2)
ax.set_ylim(0, 1.02)
limpiar(ax, f"Curva de vida típica ({n_coh} productos nacidos dentro de la ventana)",
        y="tn / pico del producto", x="edad del producto (meses)")
ax.annotate("mediana", (e[len(e)//2], curva["mediana"][len(e)//2]), xytext=(0, 10),
            textcoords="offset points", color=SERIE[0], fontsize=9, ha="center")
ax.annotate("rango intercuartil", (e[-1], curva["q75"][-1]), xytext=(-8, 8),
            textcoords="offset points", color=TINTA2, fontsize=8, ha="right")
plt.tight_layout(); plt.show()

print(curva.head(18))

In [ ]:
# ── Small multiples: 12 productos de mayor volumen ───────────────────────
# Con mas de 8 series NO se ciclan colores: se separan en paneles.
top12 = vida.sort("tn_total", descending=True).head(12)["product_id"].to_list()

fig, axes = plt.subplots(3, 4, figsize=(13, 6.5), sharex=True)
for ax, pid in zip(axes.ravel(), top12):
    d = panel.filter(pl.col("product_id") == pid).sort("m")
    xx, yy = d["m"].to_numpy(), d["tn"].to_numpy()
    ax.plot(xx, yy, color=SERIE[0], linewidth=1.6)
    ax.fill_between(xx, 0, yy, color=SERIE[0], alpha=.08)
    mp = d["m_pico"][0]
    ax.axvline(mp, color=SERIE[1], linewidth=1.2)
    cens = " (censurado izq)" if d["censura_izq"][0] else ""
    ax.set_title(f"{pid}{cens}", color=TINTA, loc="left", fontsize=9)
    ax.set_ylim(0, None); ax.grid(axis="x", visible=False)
    ax.tick_params(labelsize=7)
    ax.set_xticks([m for m in xx if m_a_periodo(m) % 100 == 1])
    ax.set_xticklabels([str(m_a_periodo(m) // 100) for m in xx if m_a_periodo(m) % 100 == 1])
fig.suptitle("Series de los 12 productos de mayor volumen  ·  línea naranja = pico",
             color=TINTA, x=.01, ha="left", fontsize=10)
plt.tight_layout(); plt.show()

## 3 — Productos similares

Similitud = **correlación de las series mensuales** sobre el tramo en que los dos
productos convivieron. Antes de correlacionar se quita la escala (z-score por
producto), así se compara la *forma* y no el tamaño.

Se filtra a productos con al menos 24 meses de historia: con menos, la correlación
es ruido. Y la pregunta que sigue es si la similitud **coincide con la jerarquía de
categorías** — si no coincide, `cat3` no está capturando el comportamiento real y un
cluster por forma de serie aportaría información nueva (eso es lo que explora el
notebook de DTW).

In [ ]:
MIN_MESES = 24
elegibles = (vida.filter((pl.col("m_muere") - pl.col("m_nace") + 1) >= MIN_MESES)
                 .sort("tn_total", descending=True))
pids = elegibles["product_id"].to_list()
print(f"{len(pids)} productos con >= {MIN_MESES} meses de historia")

# Matriz productos x meses (NaN donde el producto no estaba vivo)
meses = list(range(M_MIN, M_MAX + 1))
idx_m = {m: i for i, m in enumerate(meses)}
idx_p = {p: i for i, p in enumerate(pids)}
MAT = np.full((len(pids), len(meses)), np.nan)
sub = panel.filter(pl.col("product_id").is_in(pids)).select("product_id", "m", "tn")
for p, m, t in zip(sub["product_id"], sub["m"], sub["tn"]):
    MAT[idx_p[p], idx_m[m]] = t

# z-score por producto (ignorando NaN) -> compara forma, no escala
mu = np.nanmean(MAT, axis=1, keepdims=True)
sd = np.nanstd(MAT, axis=1, keepdims=True)
sd[sd == 0] = 1.0
Z = (MAT - mu) / sd
print(f"matriz: {Z.shape}  (productos x meses)")

In [ ]:
# Correlacion pairwise con solapamiento minimo. Se hace a mano y no con np.corrcoef
# porque cada par tiene su propia ventana de convivencia.
MIN_SOLAPE = 18
V = ~np.isnan(Z)
Z0 = np.nan_to_num(Z)

n_sol = V.astype(float) @ V.astype(float).T           # meses compartidos por par
S     = Z0 @ Z0.T                                      # suma de productos cruzados
SS    = (Z0**2) @ V.astype(float).T                    # suma de cuadrados por par
Sx    = Z0 @ V.astype(float).T                         # sumas simples

with np.errstate(invalid="ignore", divide="ignore"):
    cov = S / n_sol - (Sx / n_sol) * (Sx.T / n_sol)
    var_a = SS / n_sol - (Sx / n_sol) ** 2
    CORR = cov / np.sqrt(np.maximum(var_a, 1e-12) * np.maximum(var_a.T, 1e-12))

CORR[n_sol < MIN_SOLAPE] = np.nan
np.fill_diagonal(CORR, np.nan)
print(f"pares con solapamiento >= {MIN_SOLAPE} meses: {int(np.isfinite(CORR).sum()/2):,}")

In [ ]:
meta = {r["product_id"]: r for r in prod.select("product_id","cat1","cat2","cat3","brand").to_dicts()}

def pares_extremos(CORR, pids, n=12, mayor=True):
    iu = np.triu_indices(len(pids), k=1)
    v = CORR[iu]
    ok = np.isfinite(v)
    ii, jj, vv = iu[0][ok], iu[1][ok], v[ok]
    orden = np.argsort(-vv if mayor else vv)[:n]
    filas = []
    for k in orden:
        a, b = pids[ii[k]], pids[jj[k]]
        ma, mb = meta.get(a, {}), meta.get(b, {})
        filas.append({"A": a, "B": b, "corr": round(float(vv[k]), 3),
                      "misma_cat3": ma.get("cat3") == mb.get("cat3"),
                      "misma_brand": ma.get("brand") == mb.get("brand"),
                      "cat3_A": ma.get("cat3"), "cat3_B": mb.get("cat3")})
    return pl.DataFrame(filas)

print("=== PARES MAS PARECIDOS (correlacion positiva) ===")
print(pares_extremos(CORR, pids, 12, True))
print("\n=== PARES MAS OPUESTOS (correlacion negativa: candidatos a canibalizacion) ===")
print(pares_extremos(CORR, pids, 12, False))

In [ ]:
# ¿La similitud coincide con la jerarquia de categorias?
iu = np.triu_indices(len(pids), k=1)
v = CORR[iu]; ok = np.isfinite(v)
ii, jj, vv = iu[0][ok], iu[1][ok], v[ok]
misma3 = np.array([meta.get(pids[a],{}).get("cat3") == meta.get(pids[b],{}).get("cat3")
                   for a, b in zip(ii, jj)])
mismab = np.array([meta.get(pids[a],{}).get("brand") == meta.get(pids[b],{}).get("brand")
                   for a, b in zip(ii, jj)])

fig, ax = plt.subplots(figsize=(7.5, 3.6))
bins = np.linspace(-1, 1, 41)
ax.hist(vv[~misma3], bins=bins, density=True, color=MUDO, alpha=.55, label="distinta cat3")
ax.hist(vv[misma3],  bins=bins, density=True, histtype="step",
        color=SERIE[0], linewidth=2, label="misma cat3")
ax.axvline(0, color=EJE, linewidth=1)
limpiar(ax, "Distribución de la correlación entre pares de productos", y="densidad", x="correlación")
ax.legend(loc="upper left", fontsize=8)
plt.tight_layout(); plt.show()

print(f"correlacion media  misma cat3 : {vv[misma3].mean():+.3f}   (n={misma3.sum():,})")
print(f"correlacion media  otra  cat3 : {vv[~misma3].mean():+.3f}   (n={(~misma3).sum():,})")
print(f"correlacion media  misma marca: {vv[mismab].mean():+.3f}   (n={mismab.sum():,})")
print("\nSi las dos distribuciones se superponen, cat3 NO esta capturando la forma de la")
print("serie -> un cluster por forma (DTW) aportaria informacion que la jerarquia no tiene.")

## 4 — Canibalización

Dos mediciones independientes, porque cada una puede fallar por su lado:

**a) Correlación de *shares* dentro de la categoría.** Si dentro de `cat3` la
participación de A sube sistemáticamente cuando la de B baja, hay sustitución. Se
mira el share y no el nivel a propósito: si toda la categoría crece, dos productos
pueden subir juntos sin que eso diga nada sobre si compiten.

**b) *Event study* de lanzamientos.** Cuando entra un producto nuevo a una `cat3`,
¿qué les pasa a los que ya estaban? Se alinean todos los lanzamientos en un eje
`t = 0` y se mide el volumen de los *incumbentes* desde 6 meses antes hasta 6 meses
después, indexado a 100 en el mes previo. Es la evidencia más directa: hay un
antes y un después de un evento concreto.

In [ ]:
# ── a) Correlacion de shares dentro de cat3 ──────────────────────────────
cat_tot = panel.group_by(["cat3", "m"]).agg(pl.col("tn").sum().alias("tn_cat3"))
sh = (panel.join(cat_tot, on=["cat3", "m"], how="left")
           .with_columns(pl.when(pl.col("tn_cat3") > 0)
                           .then(pl.col("tn") / pl.col("tn_cat3"))
                           .otherwise(0.0).alias("share_cat3")))

filas = []
for c3, g in sh.filter(pl.col("product_id").is_in(pids)).group_by("cat3"):
    ps = g["product_id"].unique().to_list()
    if len(ps) < 2:
        continue
    piv = g.pivot(on="product_id", index="m", values="share_cat3").sort("m")
    A = piv.drop("m").to_numpy()
    cols = [c for c in piv.columns if c != "m"]
    if A.shape[0] < MIN_SOLAPE:
        continue
    C = np.corrcoef(np.nan_to_num(A).T)
    for a in range(len(cols)):
        for b in range(a + 1, len(cols)):
            if np.isfinite(C[a, b]):
                filas.append({"cat3": c3[0] if isinstance(c3, tuple) else c3,
                              "A": int(cols[a]), "B": int(cols[b]),
                              "corr_share": round(float(C[a, b]), 3)})

canib = pl.DataFrame(filas).sort("corr_share")
print(f"{canib.height:,} pares intra-cat3 evaluados\n")
print("=== CANDIDATOS A CANIBALIZACION (share de uno sube cuando el del otro baja) ===")
print(canib.head(15))

In [ ]:
# ── b) Event study de lanzamientos ───────────────────────────────────────
VENTANA = 6
# Lanzamientos "limpios": nacidos dentro de la ventana y con margen a ambos lados.
lanz = (vida.filter((pl.col("m_nace") > M_MIN + VENTANA) & (pl.col("m_nace") <= M_MAX - VENTANA))
            .join(prod.select("product_id", "cat3"), on="product_id", how="left"))
print(f"{lanz.height} lanzamientos con ventana completa de +-{VENTANA} meses")

curvas = []
for pid, m0, c3 in zip(lanz["product_id"], lanz["m_nace"], lanz["cat3"]):
    inc = (panel.filter((pl.col("cat3") == c3) & (pl.col("product_id") != pid)
                        & (pl.col("m_nace") < m0))["product_id"].unique().to_list())
    if len(inc) < 2:
        continue
    d = (panel.filter(pl.col("product_id").is_in(inc)
                      & pl.col("m").is_between(m0 - VENTANA, m0 + VENTANA))
              .group_by("m").agg(pl.col("tn").sum()).sort("m"))
    if d.height < 2 * VENTANA + 1:
        continue
    base = d.filter(pl.col("m") == m0 - 1)["tn"]
    if base.len() == 0 or base[0] <= 0:
        continue
    curvas.append({"pid": pid, "t": (d["m"] - m0).to_list(),
                   "idx": (100 * d["tn"] / base[0]).to_list()})

print(f"{len(curvas)} lanzamientos con incumbentes suficientes")

if curvas:
    T = np.arange(-VENTANA, VENTANA + 1)
    M = np.full((len(curvas), len(T)), np.nan)
    for i, c in enumerate(curvas):
        for t, val in zip(c["t"], c["idx"]):
            M[i, t + VENTANA] = val
    med = np.nanmedian(M, axis=0)
    q25 = np.nanpercentile(M, 25, axis=0); q75 = np.nanpercentile(M, 75, axis=0)

    fig, ax = plt.subplots(figsize=(8, 3.8))
    ax.fill_between(T, q25, q75, color=SERIE[0], alpha=.15)
    ax.plot(T, med, color=SERIE[0], linewidth=2, marker="o", markersize=5)
    ax.axvline(0, color=SERIE[1], linewidth=1.5)
    ax.axhline(100, color=EJE, linewidth=1)
    ax.annotate("lanzamiento", (0, ax.get_ylim()[1]), xytext=(5, -12),
                textcoords="offset points", color=SERIE[1], fontsize=9)
    limpiar(ax, f"Volumen de los incumbentes alrededor de un lanzamiento en su cat3  (n={len(curvas)})",
            y="índice (100 = mes previo)", x="meses respecto del lanzamiento")
    plt.tight_layout(); plt.show()

    antes = np.nanmean(med[:VENTANA]); despues = np.nanmean(med[VENTANA+1:])
    print(f"incumbentes  antes: {antes:.1f}   despues: {despues:.1f}   efecto: {despues-antes:+.1f} puntos")
    print("Negativo y sostenido = evidencia de canibalizacion.")

## 5 — Productos complementarios

Complementario = **los mismos clientes los compran en el mismo mes**, más de lo que
se esperaría si fueran independientes. Se mide con *lift* sobre la matriz
cliente-mes × producto:

```
lift(A,B) = P(A y B juntos) / (P(A) · P(B))
```

`lift > 1` es co-ocurrencia por encima del azar. Para separar complementariedad real
de simple coincidencia se pide además **correlación temporal positiva** y se marca si
comparten `cat3` — dos productos de la misma categoría que co-ocurren suelen ser
surtido, no complemento.

In [ ]:
# Incidencia binaria cliente-mes x producto
ev = (sell.filter(pl.col("tn") > 0)
          .with_columns(a_m("periodo").alias("m"))
          .select("customer_id", "m", "product_id").unique())

cm = ev.select("customer_id", "m").unique().with_row_index("fila")
ev = ev.join(cm, on=["customer_id", "m"], how="left")
prods_lift = pids                       # mismos productos elegibles de la seccion 3
ip = {p: i for i, p in enumerate(prods_lift)}
ev = ev.filter(pl.col("product_id").is_in(prods_lift))

N = cm.height
X = np.zeros((N, len(prods_lift)), dtype=np.float32)
X[ev["fila"].to_numpy(), [ip[p] for p in ev["product_id"]]] = 1.0
print(f"matriz de incidencia: {X.shape}  ({N:,} pares cliente-mes)")

CO = X.T @ X                            # co-ocurrencias
cnt = np.diag(CO).copy()
with np.errstate(invalid="ignore", divide="ignore"):
    LIFT = (CO / N) / np.outer(cnt / N, cnt / N)
np.fill_diagonal(LIFT, np.nan)
LIFT[CO < 30] = np.nan                  # pide masa minima: 30 cliente-mes juntos
print(f"pares con co-ocurrencia suficiente: {int(np.isfinite(LIFT).sum()/2):,}")

In [ ]:
iu = np.triu_indices(len(prods_lift), k=1)
lv = LIFT[iu]; ok = np.isfinite(lv)
ii, jj, lvv = iu[0][ok], iu[1][ok], lv[ok]
cv = CORR[iu][ok]                        # correlacion temporal del mismo par

orden = np.argsort(-lvv)[:400]
filas = []
for k in orden:
    a, b = prods_lift[ii[k]], prods_lift[jj[k]]
    ma, mb = meta.get(a, {}), meta.get(b, {})
    filas.append({"A": a, "B": b, "lift": round(float(lvv[k]), 2),
                  "corr_temporal": None if not np.isfinite(cv[k]) else round(float(cv[k]), 2),
                  "co_ocurr": int(CO[ii[k], jj[k]]),
                  "misma_cat3": ma.get("cat3") == mb.get("cat3")})

comp = pl.DataFrame(filas)
print("=== COMPLEMENTARIOS: lift alto + correlacion temporal positiva + distinta cat3 ===")
print(comp.filter((~pl.col("misma_cat3")) & (pl.col("corr_temporal") > 0.3)).head(15))
print("\n=== SURTIDO: lift alto pero MISMA cat3 (no es complemento, es la misma familia) ===")
print(comp.filter(pl.col("misma_cat3")).head(8))

In [ ]:
# Lift vs correlacion temporal: los cuadrantes separan complemento de sustituto.
sel = np.isfinite(cv) & (lvv > 0)
fig, ax = plt.subplots(figsize=(7, 4.6))
m3 = np.array([meta.get(prods_lift[a],{}).get("cat3") == meta.get(prods_lift[b],{}).get("cat3")
               for a, b in zip(ii, jj)])
ax.scatter(cv[sel & ~m3], lvv[sel & ~m3], s=9, color=SERIE[0], alpha=.35, label="distinta cat3")
ax.scatter(cv[sel & m3],  lvv[sel & m3],  s=9, color=SERIE[1], alpha=.55, label="misma cat3")
ax.axhline(1, color=EJE, linewidth=1); ax.axvline(0, color=EJE, linewidth=1)
ax.set_yscale("log")
limpiar(ax, "Complementariedad vs sustitución", y="lift de co-ocurrencia (log)", x="correlación temporal")
ax.annotate("complementarios", (.72, .92), xycoords="axes fraction", color=TINTA2, fontsize=9, ha="right")
ax.annotate("sustitutos", (.06, .92), xycoords="axes fraction", color=TINTA2, fontsize=9)
ax.legend(loc="lower right", fontsize=8)
plt.tight_layout(); plt.show()

## 6 — Features exportables

Todo lo anterior se condensa en un parquet a nivel **producto-mes**, listo para pegar
al dataset de `02_FE` por `(product_id, periodo)`.

Ninguna de estas columnas mira al futuro: la edad, la fase, el share y la posición
respecto del pico se calculan con información disponible **hasta ese mes**. Ojo con
una excepción importante, marcada abajo.

In [ ]:
feats = (sh.select("product_id", "periodo", "m", "edad", "largo_vida",
                   "fase", "ratio_vs_pico", "meses_desde_pico", "pend3",
                   "share_cat3", "tn_cat3", "censura_izq", "censura_der")
           .join(panel.group_by(["cat3", "m"])
                      .agg(pl.col("product_id").n_unique().alias("n_competidores_cat3"))
                      .join(prod.select("product_id", "cat3"), on="cat3", how="left")
                      .select("product_id", "m", "n_competidores_cat3"),
                 on=["product_id", "m"], how="left"))

out = DIR_OUT / "features_exploratorias.parquet"
feats.write_parquet(out)
print(f"Guardado: {out}")
print(f"{feats.height:,} filas x {feats.width} columnas")
print(feats.head(5))

### Advertencia de leakage sobre estas features

Dos de estas columnas **miran el futuro y no se pueden usar tal cual** para entrenar:

- **`ratio_vs_pico` y `meses_desde_pico`** usan el pico de *toda* la serie, incluido
  el futuro respecto del mes de la fila. En el mes 5 de un producto no se sabe que el
  pico va a estar en el mes 20.
- **`fase`** se apoya en `m_pico`, así que arrastra el mismo problema.
- **`largo_vida`** conoce la fecha de muerte.

Para explorar están perfectas: sirven para *entender* el negocio. Para entrenar hay
que recalcularlas de forma **expansiva** — el pico *hasta ese mes*, no el pico total.
La versión sin leakage es una línea distinta:

```python
pico_hasta_ahora = pl.col("tn_suave").cum_max().over("product_id")
ratio_vs_pico_ok = pl.col("tn_suave") / pico_hasta_ahora
meses_desde_pico_ok = pl.col("m") - pl.col("m").filter(...)   # argmax expansivo
```

La celda siguiente genera esa versión limpia, que es la que conviene pegarle al
dataset de entrenamiento. Si igual usás la de arriba, el control de leakage de
`03_Optuna` no la va a detectar: no es una `clase_*` ni correlaciona >0.999 con el
target, así que pasa desapercibida. Es leakage silencioso.

In [ ]:
# ── Versión SIN leakage: todo expansivo (solo pasado) ────────────────────
limpio = (panel.sort(["product_id", "m"])
    .with_columns(
        pl.col("tn_suave").cum_max().over("product_id").alias("pico_hasta_ahora"),
        pl.col("tn").cum_sum().over("product_id").alias("tn_acumulado"),
        pl.col("tn").rolling_mean(6, min_periods=1).over("product_id").alias("tn_media6"),
    )
    .with_columns(
        pl.when(pl.col("pico_hasta_ahora") > 0)
          .then(pl.col("tn_suave") / pl.col("pico_hasta_ahora"))
          .otherwise(0.0).alias("ratio_vs_pico_hist"),
        (pl.col("tn_suave") >= pl.col("pico_hasta_ahora")).alias("es_max_historico"),
    ))

# meses desde el maximo historico, expansivo
limpio = limpio.with_columns(
    pl.when(pl.col("es_max_historico")).then(pl.col("m")).otherwise(None)
      .forward_fill().over("product_id").alias("m_ultimo_max")
).with_columns((pl.col("m") - pl.col("m_ultimo_max")).alias("meses_desde_max_hist"))

feats_ok = limpio.select(
    "product_id", "periodo", "edad", "tn_acumulado", "tn_media6",
    "ratio_vs_pico_hist", "meses_desde_max_hist", "pend3", "censura_izq",
).join(sh.select("product_id", "periodo", "share_cat3", "tn_cat3"),
       on=["product_id", "periodo"], how="left")

out_ok = DIR_OUT / "features_ciclo_vida_sin_leakage.parquet"
feats_ok.write_parquet(out_ok)
print(f"Guardado: {out_ok}")
print(f"{feats_ok.height:,} filas x {feats_ok.width} columnas\n")
print(feats_ok.columns)
print("\nEsta es la que se le pega al dataset de 02_FE por (product_id, periodo).")

## Qué sigue

El notebook de **DTW** (`02_DTW_clusters.ipynb`) toma la matriz de series
normalizadas de la sección 3 y agrupa productos por **forma de la curva**, no por
categoría. La sección 3 ya deja planteada la pregunta que ese notebook responde:
si la correlación intra-`cat3` no se despega de la de pares cualesquiera, entonces
la jerarquía comercial no describe el comportamiento temporal, y un cluster por
forma le da al modelo información que hoy no tiene.